In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# NOTEBOOK 01: DATA CLEANING & FEATURE ENGINEERING
# Purpose: Handle missing values, create new features, and prepare clean data
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
pd.set_option('display.max_columns', None)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ═══════════════════════════════════════════════════════════════════════════════

df = pd.read_csv('../data/raw/customer_churn_dataset.csv')

print("="*80)
print("DATA CLEANING & FEATURE ENGINEERING")
print("="*80)
print(f"\nOriginal Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. MISSING VALUES ANALYSIS & HANDLING
# ═══════════════════════════════════════════════════════════════════════════════

print("="*80)
print("MISSING VALUES ANALYSIS")
print("="*80)

missing_stats = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percent': (df.isnull().sum().values / len(df) * 100).round(2)
})

missing_stats = missing_stats[missing_stats['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_stats) > 0:
    print("\nColumns with missing values:")
    print(missing_stats.to_string(index=False))
else:
    print("\n✓ No missing values found!")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. IMPUTATION STRATEGY
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("APPLYING IMPUTATION STRATEGY")
print("="*80)

if 'TotalCharges' in df.columns and df['TotalCharges'].isnull().sum() > 0:
    df['TotalCharges'] = df.apply(
        lambda row: row['MonthlyCharges'] * row['TenureMonths'] 
        if pd.isnull(row['TotalCharges']) and row['TenureMonths'] <= 2
        else row['TotalCharges'], 
        axis=1
    )
    if df['TotalCharges'].isnull().sum() > 0:
        df['TotalCharges'] = df.groupby(['ContractType', pd.cut(df['TenureMonths'], bins=[-1, 12, 24, 48, 100])])['TotalCharges'].transform(
            lambda x: x.fillna(x.median())
        )
    if df['TotalCharges'].isnull().sum() > 0:
        df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
    print("  ✓ TotalCharges imputed")

if 'SatisfactionScore' in df.columns and df['SatisfactionScore'].isnull().sum() > 0:
    df['SatisfactionScore'] = df.groupby(['Churn', 'ContractType'])['SatisfactionScore'].transform(
        lambda x: x.fillna(x.median())
    )
    if df['SatisfactionScore'].isnull().sum() > 0:
        df['SatisfactionScore'].fillna(df['SatisfactionScore'].median(), inplace=True)
    print("  ✓ SatisfactionScore imputed")

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
        print(f"  ✓ Catch-all imputation applied to {col}")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. BUSINESS-DRIVEN FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("CREATING BUSINESS-DRIVEN FEATURES")
print("="*80)

df['CLV_Proxy'] = df['TotalCharges']
df['ARPU'] = df['TotalCharges'] / (df['TenureMonths'] + 1)
df['ChargesRatio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)

safe_max = lambda x: x.max() if x.max() > 0 else 1

df['EngagementScore'] = (
    df['AvgMonthlyUsageGB'] / safe_max(df['AvgMonthlyUsageGB']) * 0.4 +
    df['NumProducts'] / safe_max(df['NumProducts']) * 0.3 +
    (1 - df['DaysSinceLastInteraction'] / safe_max(df['DaysSinceLastInteraction'])) * 0.3
) * 100

df['RiskScore'] = (
    df['SupportCalls'] / safe_max(df['SupportCalls']) * 0.3 +
    df['LatePayments'] / safe_max(df['LatePayments']) * 0.4 +
    df['DaysSinceLastInteraction'] / safe_max(df['DaysSinceLastInteraction']) * 0.3
) * 100

df['SatisfactionIndex'] = (
    df['SatisfactionScore'] / 5 * 0.6 +
    (1 - df['SupportCalls'] / safe_max(df['SupportCalls'])) * 0.2 +
    (1 - df['LatePayments'] / safe_max(df['LatePayments'])) * 0.2
) * 100

df['TenureCategory'] = pd.cut(df['TenureMonths'], bins=[0, 12, 24, 48, 1000], labels=['0-1 Year', '1-2 Years', '2-4 Years', '4+ Years'], include_lowest=True)
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 25, 35, 45, 55, 1000], labels=['18-25', '26-35', '36-45', '46-55', '56+'], include_lowest=True)
df['ChargesCategory'] = pd.cut(df['MonthlyCharges'], bins=[0, 30, 60, 90, 1000], labels=['Low', 'Medium', 'High', 'Premium'], include_lowest=True)
df['UsageIntensity'] = pd.cut(df['AvgMonthlyUsageGB'], bins=[0, 10, 30, 60, 1000], labels=['Light', 'Moderate', 'Heavy', 'Power User'], include_lowest=True)

service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
                'OnlineBackup', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['ServiceAdoptionScore'] = df[service_cols].apply(lambda x: sum([1 for val in x if val not in ['No', 'No internet service']]), axis=1)

df['PaymentReliability'] = 100 - (df['LatePayments'] / safe_max(df['LatePayments']) * 100)
df['InteractionRecency'] = 100 - (df['DaysSinceLastInteraction'] / safe_max(df['DaysSinceLastInteraction']) * 100)

contract_mapping = {'Month-to-month': 1, 'One year': 2, 'Two year': 3}
df['ContractValueScore'] = df['ContractType'].map(contract_mapping).fillna(1) # Fillna in case of unseen categories

df['HasPremiumServices'] = ((df['OnlineSecurity'] == 'Yes') | (df['OnlineBackup'] == 'Yes') | (df['TechSupport'] == 'Yes')).astype(int)
df['HasStreamingServices'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)

df['TotalServicesCount'] = df['NumProducts'] + df['ServiceAdoptionScore']
df['ChargesPerProduct'] = df['MonthlyCharges'] / (df['NumProducts'] + 1)
df['SupportIntensity'] = df['SupportCalls'] / (df['TenureMonths'] + 1)
df['LatePaymentRate'] = df['LatePayments'] / (df['TenureMonths'] + 1)

new_features = ['CLV_Proxy', 'ARPU', 'ChargesRatio', 'EngagementScore', 'RiskScore',
                'SatisfactionIndex', 'ServiceAdoptionScore', 'PaymentReliability',
                'InteractionRecency', 'ContractValueScore', 'HasPremiumServices',
                'HasStreamingServices', 'TotalServicesCount', 'ChargesPerProduct',
                'SupportIntensity', 'LatePaymentRate']
df[new_features] = df[new_features].fillna(0)
for cat_col in ['TenureCategory', 'AgeGroup', 'ChargesCategory', 'UsageIntensity']:
    if df[cat_col].isnull().sum() > 0:
        df[cat_col] = df[cat_col].fillna(df[cat_col].mode()[0])

print(f"\n✓ Total new features created: 20")
print(f"✓ Dataset shape after feature engineering: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ═══════════════════════════════════════════════════════════════════════════════
# 8. ENCODING CATEGORICAL VARIABLES
# ═══════════════════════════════════════════════════════════════════════════════

df_encoded = df.copy()

binary_cols = ['PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df_encoded[col] = df_encoded[col].map({'Yes': 1, 'No': 0}).fillna(0) # Safeguard

multi_level_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in multi_level_cols:
    df_encoded[col] = df_encoded[col].map({'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0}).fillna(0)

le = LabelEncoder()
ordinal_cols = ['Gender', 'ContractType', 'InternetService', 'PaymentMethod']
for col in ordinal_cols:
    df_encoded[col + '_Encoded'] = le.fit_transform(df_encoded[col].astype(str))

if 'City' in df_encoded.columns:
    city_dummies = pd.get_dummies(df_encoded['City'], prefix='City', drop_first=True)
    df_encoded = pd.concat([df_encoded, city_dummies], axis=1)

df_encoded['TenureCategory_Encoded'] = le.fit_transform(df_encoded['TenureCategory'].astype(str))
df_encoded['AgeGroup_Encoded'] = le.fit_transform(df_encoded['AgeGroup'].astype(str))
df_encoded['ChargesCategory_Encoded'] = le.fit_transform(df_encoded['ChargesCategory'].astype(str))
df_encoded['UsageIntensity_Encoded'] = le.fit_transform(df_encoded['UsageIntensity'].astype(str))

# ═══════════════════════════════════════════════════════════════════════════════
# 9. FEATURE SCALING
# ═══════════════════════════════════════════════════════════════════════════════

numerical_features = ['Age', 'TenureMonths', 'DaysSinceLastInteraction', 'MonthlyCharges', 
                      'TotalCharges', 'NumProducts', 'SupportCalls', 'SatisfactionScore', 
                      'AvgMonthlyUsageGB', 'LatePayments'] + new_features

df_encoded[numerical_features] = df_encoded[numerical_features].fillna(0)

scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[numerical_features] = scaler.fit_transform(df_encoded[numerical_features])

# ═══════════════════════════════════════════════════════════════════════════════
# 10. FINAL VALIDATION - NO NaN CHECK
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("FINAL VALIDATION")
print("="*80)

for dataset_name, dataset in [('Original+Features', df), ('Encoded', df_encoded), ('Scaled', df_scaled)]:
    nan_count = dataset.isnull().sum().sum()
    print(f"{dataset_name}: {nan_count} NaN values")
    if nan_count > 0:
        print(f"  ⚠ WARNING: {dataset_name} still has NaN!")
        print(dataset.isnull().sum()[dataset.isnull().sum() > 0])
    else:
        print(f"  ✓ {dataset_name} is perfectly clean!")

# ═══════════════════════════════════════════════════════════════════════════════
# 11. SAVE PROCESSED DATA
# ═══════════════════════════════════════════════════════════════════════════════

df.to_csv('../data/processed/customer_churn_features.csv', index=False)
df_encoded.to_csv('../data/processed/customer_churn_encoded.csv', index=False)
df_scaled.to_csv('../data/processed/customer_churn_scaled.csv', index=False)
print("\n✓ Processed datasets saved successfully!")

DATA CLEANING & FEATURE ENGINEERING

Original Dataset: 70,000 rows × 26 columns

MISSING VALUES ANALYSIS

Columns with missing values:
           Column  Missing_Count  Missing_Percent
     TotalCharges           6953             9.93
SatisfactionScore           2121             3.03

APPLYING IMPUTATION STRATEGY
  ✓ TotalCharges imputed
  ✓ SatisfactionScore imputed

CREATING BUSINESS-DRIVEN FEATURES

✓ Total new features created: 20
✓ Dataset shape after feature engineering: 70,000 rows × 46 columns

FINAL VALIDATION
Original+Features: 0 NaN values
  ✓ Original+Features is perfectly clean!
Encoded: 0 NaN values
  ✓ Encoded is perfectly clean!
Scaled: 0 NaN values
  ✓ Scaled is perfectly clean!

✓ Processed datasets saved successfully!
